# Figure 19 -- topology switching under a per-iteration rebuild

Loads `bench/results/density_reconstruction/topology_switching.json`, produced by `bench/payoff_static/topology_switching.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("density_reconstruction/topology_switching.json")
cfg, recs = art["config"], art["data"]["records"]
recs = [r for r in recs if not r.get("failed")]
if not recs:
    raise SystemExit("topology_switching.json has no successful rows")

def rate(r, name):
    return (r.get("switch_summary") or {}).get("intensive", {}).get("mean", {}).get(name)

fig, axes = style.figure(width=style.TWO_COL, height=5.0, ncols=2, nrows=2)

# (a) THE METRIC PANEL: intensive rates resolve, the extensive one saturates. #
ax = axes[0][0]
base_lr = min(r["learning_rate"] for r in recs)
sel = sorted((r for r in recs if r["learning_rate"] == base_lr
              and r["rebuild_cadence"] == 1), key=lambda r: r["N"])
Ns = [r["N"] for r in sel]
for i, (name, label) in enumerate((
        ("near_set_churn", "near-field source set (intensive)"),
        ("near_pair_churn", "near leaf-pair set"),
        ("leaf_churn", "leaf membership"),
        ("slot_churn", "Morton slot"))):
    values = [rate(r, name) for r in sel]
    if all(v is None for v in values):
        continue
    ax.plot(Ns, [v if v is not None else np.nan for v in values],
            marker=style.MARKERS[i % len(style.MARKERS)],
            color=style.CATEGORICAL[i % len(style.CATEGORICAL)], label=label)
ax.plot(Ns, [(r["switch_summary"] or {}).get("extensive", {}).get("switch_rate")
             for r in sel], marker="x", ls=":", color=style.INK,
        label="extensive 'anything changed?'")
ax.set_xscale("log"); ax.set_ylim(-0.04, 1.10)
ax.set_xlabel("$N$"); ax.set_ylabel("churn per rebuild")
style.finish(ax, legend=True, legend_kwargs={"loc": "lower right", "fontsize": 5.2})

# (b) cadence: how stale may the interaction list be? --------------------- #
ax = axes[0][1]
for i, n in enumerate(sorted({r["N"] for r in recs})):
    sel = sorted((r for r in recs if r["N"] == n and r["learning_rate"] == base_lr),
                 key=lambda r: r["rebuild_cadence"])
    if len(sel) < 2:
        continue
    ks = [r["rebuild_cadence"] for r in sel]
    ax.plot(ks, [rate(r, "near_set_churn") for r in sel],
            marker=style.MARKERS[i % len(style.MARKERS)],
            color=style.CATEGORICAL[i % len(style.CATEGORICAL)],
            label="N=%d, interaction set" % n)
    base = sel[0]["final_loss"]
    ax.plot(ks, [r["final_loss"] / base for r in sel], marker="x", ls="--",
            color=style.CATEGORICAL[i % len(style.CATEGORICAL)],
            label="N=%d, final loss / loss(k=1)" % n)
    # The loss jump at a rebuild, in units of what one step achieves. Below one
    # means switching costs less than a step gains -- which is the claim.
    ratios = [(r.get("loss_continuity") or {}).get("jump_over_step_improvement")
              for r in sel]
    if any(v is not None for v in ratios):
        ax.plot(ks, [v if v is not None else np.nan for v in ratios], marker="+",
                ls=":", color=style.CATEGORICAL[i % len(style.CATEGORICAL)],
                label="N=%d, jump / step improvement" % n)
ax.set_xscale("log", base=2)
ax.set_xlabel("rebuild cadence $k$"); ax.set_ylabel("churn, and relative final loss")
style.finish(ax, legend=True, legend_kwargs={"loc": "center left", "fontsize": 5.2})

# (c) loss continuity across a switch, against the progress a step makes --- #
ax = axes[1][0]
pts = [(r["N"], r["rebuild_cadence"], r["loss_continuity"]["median_relative_jump"],
        r["loss_continuity"]["max_relative_jump"])
       for r in recs if r.get("loss_continuity")
       and r["loss_continuity"].get("median_relative_jump") is not None]
if pts:
    for i, n in enumerate(sorted({p[0] for p in pts})):
        sub = sorted((p for p in pts if p[0] == n), key=lambda p: p[1])
        ax.plot([p[1] for p in sub], [p[2] for p in sub],
                marker=style.MARKERS[i % len(style.MARKERS)],
                color=style.CATEGORICAL[i % len(style.CATEGORICAL)],
                label="N=%d, median" % n)
        ax.plot([p[1] for p in sub], [p[3] for p in sub], marker="x", ls=":",
                color=style.CATEGORICAL[i % len(style.CATEGORICAL)],
                label="N=%d, max" % n)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("rebuild cadence $k$")
ax.set_ylabel("loss jump at a rebuild, relative")
style.finish(ax, legend=True, legend_kwargs={"loc": "upper left", "fontsize": 5.2})

# (d) the contract, and how far the gradient moves across a rebuild -------- #
# NOT a finite difference straddling the switch: that divides a topology-induced
# loss offset by 2*eps, diverges as eps -> 0, and read 5e3 in the first version
# of this figure. The artifact keeps its eps-scaling demonstration; what belongs
# on a plot is the gradient difference, which is bounded and comparable to the
# Yggdrax exactness result.
ax = axes[1][1]
fd = [r for r in recs if r.get("fd_agreement")]
if fd:
    x = np.arange(len(fd))
    ax.plot(x, [r["fd_agreement"]["pinned_median_rel"] for r in fd],
            marker=style.MARKERS[0], color=style.ENTITY["positions"],
            label="FD vs autodiff, pinned epoch")
    delta = [r["fd_agreement"].get("gradient_delta_rel_across_rebuild") for r in fd]
    if any(v is not None for v in delta):
        ax.plot(x, [v if v is not None else np.nan for v in delta],
                marker=style.MARKERS[2], ls="--", color=style.CATEGORICAL[2],
                label="gradient change across a rebuild")
    ax.set_xticks(x)
    ax.set_xticklabels(["%d/k%d" % (r["N"], r["rebuild_cadence"]) for r in fd],
                       rotation=60, fontsize=4.6)
ax.set_yscale("log")
ax.set_ylabel("relative difference")
style.finish(ax, legend=True, legend_kwargs={"loc": "upper left", "fontsize": 5.2})

fig.tight_layout()
style.footer(fig, "%s, leaf %d, order %d, theta %.2f, %d iterations, lr %g" % (
    art["meta"]["device_kind"], cfg["leaf_size"], cfg["order"], cfg["theta"],
    cfg["iterations"], base_lr))
style.save(fig, str(FIG_DIR / "fig19_topology_switching.pdf"))


## Caption


What a per-iteration tree rebuild does to a gradient descent -- the section's
second substantive result, and the first high-dimensional evidence that
per-iteration rebuilds do not obstruct convergence for an inference objective
through an FMM. **(a)** Why the rate is reported per particle. The extensive
"did the discrete structure change at all?" counter reads exactly one at every
$N$ and carries no information; the intensive rates resolve the structure. The
top curve to read is the near-field *source set* churn -- the fraction of the
sources reaching a given particle by direct summation that changed -- which is
the interaction-list question, and which the leaf-pair and Morton-slot rates
both overstate badly. **(b)** Cadence: the interaction list becomes almost
entirely stale as $k$ grows while the final loss does not move, so $k>1$ is
usable, and it is this measurement rather than any appeal to ordering stability
that says so. **(c)** The loss discontinuity at a rebuild boundary, measured at
*fixed positions* under the outgoing and incoming topologies so it isolates the
topology change from the optimiser's step. **(d)** Finite differences against
autodiff within one topology epoch, where they agree to $\sim10^{-9}$ -- the
fixed-topology contract of Sect.~2 measured rather than argued -- beside the
relative change in the *gradient itself* when the topology is rebuilt at the
same positions. The pinned arm pins the interaction-list selection as well as
the tree; pinning only the tree leaves a residual that reads as a gradient bug
and is not one. A finite difference whose two evaluations straddle a switch is
deliberately not plotted: it divides a topology-induced loss offset by the step
size, so it diverges as the step shrinks and reports the step rather than the
pipeline. Cited alongside, and not confirmed by, the low-dimensional Yggdrax
precedent.
